# 第 2 周练习 —— 带聊天记录的问答聊天机器人

## 练习目标

做一个**代码讲解助手**：用户可贴代码或提问；助手用初学者友好的方式解释，并在可能时给示例与更优写法。界面用 Gradio `Blocks`，支持：

- 下拉切换本地模型（`llama3.2:1b` / `phi`）
- 多轮历史（`gr.State` + `Chatbot`）
- 流式回复（边生成边刷新气泡）

## 和本课 Week 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Ollama OpenAI 兼容接口 | `OLLAMA_BASE_URL` + `OpenAI(...)` |
| `messages` = system + 历史 + user | 在 `chat_fn` 里手动拼装 |
| 流式 `stream=True` | 累加 `delta.content` 并 `yield` |
| Gradio Blocks / State | 按钮绑定、清空历史 |

## 怎么跑

1. 本机 Ollama 运行中，并已拉取下拉框里的模型
2. 运行代码格 → 浏览器打开 Gradio → 选模型 → 输入问题或粘贴代码 → Send


In [ ]:
# ========== 带历史的代码问答机器人：Ollama + Gradio Blocks ==========

# 从 openai 导入 OpenAI：经兼容 /v1 调用本地 Ollama
from openai import OpenAI
# 导入 gradio：用 Blocks 自定义布局（下拉、聊天区、按钮）
import gradio as gr

# Ollama 的 OpenAI 兼容基址（URL 字符串必须原样保留）
OLLAMA_BASE_URL = "http://localhost:11434/v1"
# 创建客户端；api_key 占位（Ollama 本地一般不校验）
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# system_prompt：代码导师人设（英文原文保留，决定回答风格）
system_prompt = """
You are a professional software coding master. 
Explain code in a simple beginner-friendly way.
Give examples and suggest better alternatives if possible.
"""

# chat_fn：发送按钮回调；message 当前输入，history 元组列表，model 下拉选中的模型名
def chat_fn(message, history, model):
    # history 可能为 None（首轮）；归一成空列表，避免后面 for 循环报错
    history = history or []

    
    # 每轮从 system 开始重建 messages（无状态服务端，历史由客户端维护）
    messages = [{"role": "system", "content": system_prompt}]

    # 把旧轮次 (user, assistant) 元组展开成 role/content，接到 messages
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})

    # 追加本轮用户问题
    messages.append({"role": "user", "content": message})

   
    # 流式调用所选模型；model 来自 Dropdown，勿写死
    stream = ollama.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )

    # reply：累积助手文本；每次 yield 同时更新 Chatbot 与 State
    reply = ""
    for chunk in stream:
        reply += chunk.choices[0].delta.content or ""
        # 两个输出相同：chatbot 显示用、state 持久化用（均为 history + 本轮元组）
        yield history + [(message, reply)], history + [(message, reply)]



# Blocks：比 Interface 更灵活，可放多个控件与事件
with gr.Blocks() as app:
    # 页面标题（Markdown 字符串保持原样）
    gr.Markdown("# 💻 AI Code Assistant")

    # 模型下拉：选项与默认值均为本地 Ollama 模型名
    model = gr.Dropdown(
        ["llama3.2:1b", "phi"],
        value="llama3.2:1b",
        label="Choose Model"
    )

    # 聊天气泡区（显示 history 元组列表）
    chatbot = gr.Chatbot()

    # 用户输入框；placeholder / label 保持英文原样
    msg = gr.Textbox(
        placeholder="Ask a question or paste code...",
        label="Your Input"
    )

    # 发送与清空按钮
    send_btn = gr.Button("Send")
    clear_btn = gr.Button("Clear Chat")

    # State：在多次点击之间保存 history（初始为空列表）
    state = gr.State([])

    
    # Send：把 msg、state、model 喂给 chat_fn；结果写回 chatbot 与 state
    send_btn.click(
        chat_fn,
        inputs=[msg, state, model],
        outputs=[chatbot, state]
    )

    
    # Clear：lambda 返回两个空列表，同时清空界面与状态
    clear_btn.click(
        lambda: ([], []),
        inputs=[],
        outputs=[chatbot, state]
    )

# 启动 Gradio 应用（笔记本内会给出本地 URL）
app.launch()
